# ShopSense AI - Quick Start

Get up and running with ShopSense AI recommendation engine in 5 minutes.

**Prerequisites:**
- Docker running
- Services started (`docker compose up`)
- API available at http://localhost:8000
- Dataset downloaded (`python scripts/download_data.py --sample-size 500`)

## 1. Test API Health

In [ ]:
import requests
import json

api_url = "http://localhost:8000/api/v1"

# Check health
health = requests.get(f"{api_url}/health/").json()
print(json.dumps(health, indent=2))

## 2. Get Similar Products

In [ ]:
# Get similar products for Sony WH-1000XM5
product_id = "B08N5WRWNW"
k = 8
method = "hybrid"

url = f"{api_url}/products/{product_id}/similar/"
params = {"k": k, "method": method}

response = requests.get(url, params=params)
result = response.json()

print(f"Found {len(result['similar_products'])} similar products in {result['latency_ms']}ms")
print(f"Cache hit: {result['cache_hit']}")
print("\nTop 3 recommendations:")
for product in result['similar_products'][:3]:
    print(f"  #{product['rank']}: {product['title']} ({product['match_percent']}%)")

## 3. Compare Methods

In [ ]:
import time

methods = ["tfidf_only", "embedding_only", "hybrid"]
results = {}

for method in methods:
    start = time.time()
    resp = requests.get(url, params={"k": 8, "method": method})
    elapsed = (time.time() - start) * 1000
    
    data = resp.json()
    results[method] = {
        "latency": data["latency_ms"],
        "cache_hit": data["cache_hit"],
        "top_score": data["similar_products"][0]["similarity_score"]
    }

# Display comparison
print("Method Comparison:\n")
for method, stats in results.items():
    print(f"{method:20} | Latency: {stats['latency']:4}ms | Score: {stats['top_score']:.3f}")

## 4. Batch Processing

In [ ]:
# Process multiple products at once
product_ids = ["B08N5WRWNW", "B09G9FPHY6", "B08DHPZX6K"]

batch_response = requests.post(
    f"{api_url}/batch-similar/",
    json={"product_ids": product_ids, "k": 5}
)

if batch_response.status_code == 200:
    batch_data = batch_response.json()
    print(f"Processed {len(product_ids)} products")
    print(f"Total time: {batch_data.get('_metadata', {}).get('latency_ms', 'N/A')}ms")
else:
    print(f"Error: {batch_response.status_code}")

## 5. Performance Metrics

In [ ]:
# Get engine statistics
stats = requests.get(f"{api_url}/stats/").json()

print("Engine Statistics:")
print(f"  Total products: {stats['n_products']:,}")
print(f"  Vocabulary size: {stats['tfidf_features']:,}")
print(f"  Embedding dimension: {stats['embedding_dim']}")
print(f"  TF-IDF sparsity: {stats['tfidf_sparsity']:.1%}")

## 6. Benchmark Latency

In [ ]:
import time
import numpy as np

# Benchmark 10 requests to measure caching
latencies = []

for i in range(10):
    start = time.time()
    resp = requests.get(url, params={"k": 8, "method": "hybrid"})
    latencies.append((time.time() - start) * 1000)

print(f"Latency Benchmark (10 requests):")
print(f"  Min:  {min(latencies):.1f}ms")
print(f"  Mean: {np.mean(latencies):.1f}ms")
print(f"  Max:  {max(latencies):.1f}ms")
print(f"\nFirst request (cache miss): {latencies[0]:.1f}ms")
print(f"Remaining (cache hits):    {np.mean(latencies[1:]):.1f}ms average")
print(f"Speedup from caching:      {latencies[0] / np.mean(latencies[1:]):.1f}x faster")

## Next Steps

1. **Explore Dashboard**: Open http://localhost:8501 in browser
2. **Read Documentation**: See [README.md](../README.md) and [ARCHITECTURE.md](../ARCHITECTURE.md)
3. **Run Full Tests**: `pytest tests/ -v`
4. **Study ML Pipeline**: Check [01_data_exploration.ipynb](01_data_exploration.ipynb)